In [19]:
PRIVATE_LIMITED_LOOKUP = {
    # ===============================
    # Standard spaced forms
    # ===============================
    "PRIVATE LIMITED": "PRIVATE LIMITED",
    "PRIVATE LTD": "PRIVATE LIMITED",
    "PVT LIMITED": "PRIVATE LIMITED",
    "PVT LTD": "PRIVATE LIMITED",

    # ===============================
    # No-space / joined forms
    # ===============================
    "PRIVATELIMITED": "PRIVATE LIMITED",
    "PRIVATELTD": "PRIVATE LIMITED",
    "PVTLIMITED": "PRIVATE LIMITED",
    "PVTLTD": "PRIVATE LIMITED",

    # ===============================
    # Dot / comma separated
    # ===============================
    "PRIVATE.LIMITED": "PRIVATE LIMITED",
    "PRIVATE,LIMITED": "PRIVATE LIMITED",
    "PRIVATE.LTD": "PRIVATE LIMITED",
    "PRIVATE,LTD": "PRIVATE LIMITED",
    "PVT.LIMITED": "PRIVATE LIMITED",
    "PVT,LIMITED": "PRIVATE LIMITED",
    "PVT.LTD": "PRIVATE LIMITED",
    "PVT,LTD": "PRIVATE LIMITED",

    # ===============================
    # Trailing punctuation
    # ===============================
    "PVT.LTD.": "PRIVATE LIMITED",
    "PVT.LTD,": "PRIVATE LIMITED",
    "PVTLTD.": "PRIVATE LIMITED",
    "PVTLTD,": "PRIVATE LIMITED",

    # ===============================
    # Bracketed PRIVATE + LTD (NO space)
    # e.g. Integera Software Services(Pvt)Ltd
    # ===============================
    "(PVT)LTD": "PRIVATE LIMITED",
    "(PVT)LIMITED": "PRIVATE LIMITED",
    "(PRIVATE)LTD": "PRIVATE LIMITED",
    "(PRIVATE)LIMITED": "PRIVATE LIMITED",

    "(PVT).LTD.": "PRIVATE LIMITED",
    "(PVT).LTD": "PRIVATE LIMITED",
    "(PVT)LTD.": "PRIVATE LIMITED",
    "(PVT).LIMITED.": "PRIVATE LIMITED",
    "(PRIVATE).LTD.": "PRIVATE LIMITED",
    "(PRIVATE).LIMITED.": "PRIVATE LIMITED",

    # ===============================
    # Bracketed PRIVATE + LTD (WITH space)
    # ===============================
    "(PVT) LTD": "PRIVATE LIMITED",
    "(PRIVATE) LTD": "PRIVATE LIMITED",

    # ===============================
    # OCR / typo combinations
    # ===============================
    "PRIVATELIMTED": "PRIVATE LIMITED",
    "PRIVTELIMITED": "PRIVATE LIMITED",
    "PVTLIMTED": "PRIVATE LIMITED",
    "PVT LIMTED": "PRIVATE LIMITED",
}



PRIVATE_LOOKUP = {
    # Standard
    "PRIVATE,": "PRIVATE",
    "PRIVATE.": "PRIVATE",
    "PRIVATE": "PRIVATE",
    "PVT," : "PRIVATE",
    "PVT.": "PRIVATE",
    "PVT": "PRIVATE",

    # OCR / typos
    "PRIVTE": "PRIVATE",
    "PRVATE": "PRIVATE",
    # " PRIATE": " PRIVATE",
    "PRITVATE": "PRIVATE",

    # Joined / spacing issues
    # " PV T": " PRIVATE",
    "PRIV ATE": "PRIVATE",

    # Brackets / noise
    "(PRIVATE),": "PRIVATE",
    "(PRIVATE).": "PRIVATE",
    "(PRIVATE)": "PRIVATE",
    "(P)," : "PRIVATE",
    "(P)." : "PRIVATE",
    "(P)" : "PRIVATE",
}

LIMITED_LOOKUP = {
    # Standard
    "LIMITED," : "LIMITED",
    "LIMITED." : "LIMITED",
    "LIMITED": "LIMITED",
    "LTD.": "LIMITED",
    "LTD": "LIMITED",

    # OCR / typos
    "LIMTED,": "LIMITED",
    "LIMTED.": "LIMITED",
    "LIMTED": "LIMITED",
    "LIMTD": "LIMITED",
    "LMTD": "LIMITED",
    "L1TD": "LIMITED",
    "LT0": "LIMITED",

    # Joined / spacing issues
    "L T D": "LIMITED",
    "LTD ": "LIMITED",

    # Noise
    "(LTD),": "LIMITED",
    "(LTD).": "LIMITED",
    "(LTD)": "LIMITED",
}

LLP_LOOKUP = {
    # Standard
    "LLP" : "LLP",
    "LIMITED LIABILITY PARTNERSHIP": "LLP",
    "LIMITED LIABILITY PARTNERSHIP.": "LLP",
    "LIMITED LIABILITY PARTNERSHIP FIRM": "LLP",

    # Common abbreviations
    # "L L P": "LLP",
    # "L.L.P.": "LLP",
    # "L.L.P": "LLP",
    "LLP.": "LLP",
    "LLP," : "LLP",

    # OCR / typo variants (very common in scanned orders)
    # "LL P": "LLP",
    # "LLR": "LLP",
    # "ILP": "LLP",
    # "LIP": "LLP",
    # "L L R": "LLP",

    # Noise / brackets
    "(LLP).": "LLP",
    "(LLP),": "LLP",
    "(LLP)": "LLP",

}

INDIA_LOOKUP = {
    # Standard
    "INDIA": "INDIA",

    # Abbreviations
    # "IND": "INDIA",
    # "IND.": "INDIA",

    # Common OCR / typo variants
    # "INDA": "INDIA",
    # "INDlA": "INDIA",   # lowercase L instead of I
    # "1NDIA": "INDIA",   # 1 instead of I
    # "IN DIA": "INDIA",

    # Joined / spacing issues
    # "I N D I A": "INDIA",

    # Brackets / noise
    "(INDIA)." : "INDIA",
    "(INDIA)," : "INDIA",
    "(INDIA)": "INDIA",
    "(I)." : "INDIA",
    "(I)," : "INDIA",
    "(I)": "INDIA",
    }





In [20]:
######################### Function wise processing for Nexensus ####################################################
import os
import time
import random
import re
from fuzzywuzzy import fuzz
import pandas as pd
import requests

def normalize_company_suffix(name: str) -> str:
    if not name:
        return name

    name_up = " ".join(name.upper().split())

    # Replace PRIVATE LIMITED first
    for key in sorted(PRIVATE_LIMITED_LOOKUP, key=len, reverse=True):
        name_up = name_up.replace(key, PRIVATE_LIMITED_LOOKUP[key])

    # Replace PRIVATE first
    for key in sorted(PRIVATE_LOOKUP, key=len, reverse=True):
        name_up = name_up.replace(key, PRIVATE_LOOKUP[key])

    # Replace LIMITED next
    for key in sorted(LIMITED_LOOKUP, key=len, reverse=True):
        name_up = name_up.replace(key, LIMITED_LOOKUP[key])

    # LLP
    for key in sorted(LLP_LOOKUP, key=len, reverse=True):
        name_up = name_up.replace(key, LLP_LOOKUP[key])

    # LLP
    for key in sorted(INDIA_LOOKUP, key=len, reverse=True):
        name_up = name_up.replace(key, INDIA_LOOKUP[key])

    return name_up


def find_ratio(output_dict):
    input_name = output_dict['input']
    corrected_name = output_dict['corrected_name']
    if isinstance(corrected_name, float) or corrected_name is None:
        # ratio_list.append(0)
        output_dict['ratio'] = 0
        # output_dict = {"input": input_name, "cin": cin, "corrected_name": corrected_name, "status" : status, "ratio": 0}
    else:
        if fuzz.ratio(input_name.lower().strip(), corrected_name.lower().replace("(old name)", "").strip()) != 100:
            ratio = fuzz.ratio(normalize_company_suffix(input_name).strip(), normalize_company_suffix(corrected_name.replace("(old name)", "").strip()))
            # ratio_list.append(fuzz.ratio(normalize_company_suffix(row.input).strip(), normalize_company_suffix(row.corrected_name.replace("(old name)", "").strip())))
            output_dict['ratio'] = ratio
            # output_dict = {"input": input_name, "cin": cin, "corrected_name": corrected_name, "status" : status, "ratio": ratio}
        else:
            ratio = fuzz.ratio(input_name.lower().strip(), corrected_name.lower().replace("(old name)", "").strip())
            # ratio_list.append(fuzz.ratio(row.input.lower().strip(), row.corrected_name.lower().replace("(old name)", "").strip()))
            output_dict['ratio'] = ratio
            # output_dict = {"input": input_name, "cin": cin, "corrected_name": corrected_name, "status" : status, "ratio": ratio}
    return output_dict

def create_session():
    session = requests.Session()


    login_url = "https://nexensus.in/web/home.nex"

    payload = {
        "userName": "kripalsingh.parwal@nexensus.com",
        "password": "Kripal@123"
    }

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/143.0.0.0 Safari/537.36"
        ),
        "Referer": "https://nexensus.in/",
    }

    response = session.post(
        login_url,
        headers=headers,
        data=payload,   # form-data login
        timeout=30
    )

    print(response.status_code)
    print(session.cookies.get_dict())
    return session

session = create_session()

all_data = []

def extract_after_manager(simplified: str):
    # Split on "manager" (case-insensitive), only once
    parts = re.split(r'manager', simplified, flags=re.IGNORECASE)
    
    if len(parts) < 2:
        return simplified  # or return None if you prefer
    
    result = parts[1]

    # Remove leading spaces, commas, periods
    result = result.lstrip(" ,.")

    return result.strip()

def process_company_nexensus(input_name):
    # print("input :", input_name)
    try:
        r = session.get(
            "https://nexensus.in/web/nexen/search/searchByNamePrefix?name={}".format(input_name.replace(" ", "%20")),
            timeout=30
        )
        # print("output :", len(r.json()))
        # print()
        time.sleep(0.5)
        output_json = r.json()
        print("output_json 1:", output_json)
        for element in output_json:
            try:
                corrected_name = element['name']
            except:
                corrected_name = None
            corrected_name = re.sub(r"\s+", " ", corrected_name).strip()
            try:
                cin = element['cin'].strip()
            except:
                cin = None
            try:
                status = element['status'].strip()
            except:
                status = None

            output_dict = {"input": input_name, "cin": cin, "corrected_name": corrected_name, "status" : status}
            output_dict = find_ratio(output_dict)
            # print("output_dict", output_dict)
            if output_dict['ratio'] == 100 and output_dict['status'] == 'Active':
                all_data.append(output_dict)
                print("coming here")
                return output_dict
            else:
                if output_dict not in all_data:
                    all_data.append(output_dict)
    except Exception as e:
        print("❌ Failed — retrying with simplified name:", e)
        try:
            simplified = (
                input_name.lower()
                .replace("(p) limited.", "")
                .replace("(p) limited", "")  
                .replace("p limited.", "")  
                .replace("p Limited", "")  
                .replace("(p) ltd.", "")    
                .replace("(p).ltd.", "")
                .replace("p.Ltd.", "")  
                .replace("p Ltd.", "")  
                .replace("pLtd.", "")  
                .replace("(p) Ltd", "")
                .replace("(p)Ltd", "")
                .replace("pLtd", "")  
                .replace("p Ltd", "")          
                .replace("limited.", "")
                .replace("private.", "")
                .replace("limited", "")
                .replace("private", "")
                .replace("pvt.", "")
                .replace("ltd.", "")
                .replace("pvt", "")
                .replace("ltd", "")
                .replace("llp.", "")
                .replace("llp", "")
                .replace("messers", "")
                .replace("messars", "")
                .replace("m/s", "")
                .replace("()", "")
                # .replace("A.O.,", "")
                .strip()
                .rstrip(',')
                .lstrip(',')
                .lstrip(".")
            )
            if len(simplified) == 3:
                simplified = simplified + " "
            simplified = extract_after_manager(simplified)
            if "officer," in simplified:
                simplified = simplified.split("officer,")[1].strip()
            elif "officer ," in simplified:
                simplified = simplified.split("officer,")[1].strip()

            r = session.get(
                "https://nexensus.in/web/nexen/search/searchByNamePrefix?name={}".format(simplified.replace(" ", "%20")),
                timeout=30
            )
            # print("output :", len(r.json()))
            # print()
            time.sleep(0.5)
            output_json = r.json()
            print("output_json 2:", output_json)
            for element in output_json:
                try:
                    corrected_name = element['name']
                except:
                    corrected_name = None
                corrected_name = re.sub(r"\s+", " ", corrected_name).strip()
                try:
                    cin = element['cin'].strip()
                except:
                    cin = None
                try:
                    status = element['status'].strip()
                except:
                    status = None

                output_dict = {"input": input_name, "cin": cin, "corrected_name": corrected_name, "status" : status}
                output_dict = find_ratio(output_dict)
                # print("output_dict", output_dict)
                if output_dict['ratio'] == 100 and output_dict['status'] == 'Active':
                    all_data.append(output_dict)
                    print("coming here")
                    return output_dict
                else:
                    if output_dict not in all_data:
                        all_data.append(output_dict)
        except:
            output_dict = {"input": input_name, "cin": None, "corrected_name": None, "status" : None, "ratio": None}
            all_data.append(output_dict)
            return output_dict
    # return all_data
    # return output_dict
    # return {"input": input_name, "corrected_name": None, "cin": None, "status": None, 'ratio': None}

input_list = pd.read_excel(r"D:\Nexensus_Projects\ToflerAndZuba\input_folder\Book10.xlsx")['c_name'].drop_duplicates().to_list()[:10]
for input_name in input_list[:]:
    print("input :", input_name)
    print("output :", process_company_nexensus(input_name))
    # idx += 1

200
{'JSESSIONID': 'iYO7kL7xNNuddp-TBXGL9xrXI3-nBn_-OAFgaNj6.localhost'}
input : Compuage Infocom Limited
output_json 1: [{'id': 566964, 'encryptedId': '566964', 'name': 'Compuage Infocom Limited', 'selected': '566964', 'cin': 'L99999MH1999PLC135914', 'status': 'Active ', 'oldCompanyNames': 'Compuage Infocom Limited'}]
coming here
output : {'input': 'Compuage Infocom Limited', 'cin': 'L99999MH1999PLC135914', 'corrected_name': 'Compuage Infocom Limited', 'status': 'Active', 'ratio': 100}
input : Ncml Mktyard Private Limited
output_json 1: [{'id': 1657366, 'encryptedId': '1657366', 'name': 'Ncml Mktyard Private Limited', 'selected': '1657366', 'cin': 'U51901HR2017PTC067265', 'status': 'Active ', 'oldCompanyNames': None}]
coming here
output : {'input': 'Ncml Mktyard Private Limited', 'cin': 'U51901HR2017PTC067265', 'corrected_name': 'Ncml Mktyard Private Limited', 'status': 'Active', 'ratio': 100}
input : Viom Infra Ventures Limited
output_json 1: [{'id': 912541, 'encryptedId': '912541', 

In [11]:
# === Output File ===
output_path = "testdata.xlsx"

if all_data:
    final_df = pd.DataFrame(all_data)
    # final_df = pd.read_excel(r"D:\Nexensus_Projects\ToflerAndZuba\Book7_data.xlsx")

    # --- 1) Filter Active + ratio == 100 ---
    df_100_active = final_df[(final_df['status'] == 'Active') & (final_df['ratio'] == 100)]

    # --- 2) Remove rows from df_100_active based on input column ---
    df_remaining = final_df[~final_df['input'].isin(df_100_active['input'])]

    # --- 3) Filter 80Plus from remaining ---
    df_80_plus = df_remaining[df_remaining['ratio'] >= 80]

    # --- 4) Remaining after removing 80Plus ---
    df_remaining_final = df_remaining[~df_remaining['input'].isin(df_80_plus['input'])]

    # --- Save to Excel with multiple sheets ---
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        final_df.to_excel(writer, sheet_name='allData', index=False)
        df_100_active.to_excel(writer, sheet_name='100Active', index=False)
        df_80_plus.to_excel(writer, sheet_name='80Plus', index=False)
        df_remaining_final.to_excel(writer, sheet_name='remaining', index=False)

    print(f"🎉 All done! Final file saved to {output_path}")

🎉 All done! Final file saved to testdata.xlsx
